In [2]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("Scaling-book"):
        !git clone https://github.com/arjuns238/Scaling-book.git
    %cd Scaling-book/Addition_Transformer
    !pip install -q flax optax

Cloning into 'Scaling-book'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 67 (delta 34), reused 45 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 170.60 KiB | 7.11 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/Scaling-book/Addition_Transformer


In [3]:
import jax
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

In [ ]:
# # chose configs - ~10M params
from model import *
from data import build_dataset, generate_split, Dataloader
import numpy as np
from config import Config
base_cfg = Config(
    d_model=384,
    ffw_multiplier=8/3,
    num_layers=6,
    query_heads=6,
    kv_heads=6,
    key_dim=64,
    vocab_size=16,
    batch_size = 256,
    num_experts = 8,   # > 0 selects the moe ffw; 0 would run the dense path
    top_k = 2,
    lb_factor = 0.01,
    dtype=jnp.bfloat16,
    lr = 1e-3,
    num_epochs = 15,
)

In [4]:
from utils import *

# Precompute (d_model, num_layers, N) once so we can pick sizes by N.
def size_table(base, SIZE_LADDER):
    table = []
    for dm, nl in SIZE_LADDER:
        w = Weights.init(make_cfg(base, dm, nl), jax.random.key(0))
        table.append((dm, nl, count_params(w)))
    return table  # list of (d_model, num_layers, N)

The smallest model is N=100,416. 

With N_min = 100416 and SUP_PER_EX ≈ 4 ( supervised tokens per example —, it's roughly the answer digits + EOS, so ~4 for the 2-and-3-digit-dominated draw).

The hard ceiling is distinct non-fundamental pairs: ~999,900. Fundamental pairs are defined as the cases with 1-digit addition as they are the fundamentals for all other cases. These are excluded from loss calculation.

To stay single-epoch (all fresh data, clean Chinchilla regime), the largest feasible budget for the smallest model:

C_max <= 6 * N_min * SUP_PER_EX * POOL_EXAMPLES
      =  6 * 100416 * 4 * 999900
      ≈  2,409,743,001,600
      ≈  2.4e12

In [5]:
from data import create_example
def build_full_pool(max_seq_len, max_n=1000, seed=0, exclude_both_1digit=True):
    pairs = []
    for a in range(max_n):
        for b in range(max_n):
            if exclude_both_1digit and a < 10 and b < 10:
                continue
            pairs.append((a, b))
    rng = np.random.default_rng(seed)
    rng.shuffle(pairs)  # so any prefix is a representative sample, not a-major

    n = len(pairs)
    tokens = np.empty((n, max_seq_len), dtype=np.int32)
    masks  = np.empty((n, max_seq_len), dtype=np.int32)
    for i, (a, b) in enumerate(pairs):
        t, m = create_example(a, b, max_seq_len)   # your fn
        tokens[i] = t
        masks[i]  = m
    return tokens, masks

In [9]:
def budget_ceiling(N_min, sup_per_ex, n_unique_train):
    """Largest C at which the smallest model still trains single-epoch."""
    D_max_tokens = n_unique_train * sup_per_ex
    return 6 * N_min * D_max_tokens

def examples_for(C, N, sup_per_ex):
    return int(round(C / (6 * N * sup_per_ex)))

In [10]:
# ----------------------------------------------------------------------
# 5. Corner check: 4 extremes -> go/no-go before the full sweep.
# ----------------------------------------------------------------------
def corner_check(base_cfg, tbl, budgets, train_tok, train_mask,
                 val_tok, val_mask, sup_per_ex, batch_size):
    n_train = len(train_tok)
    smallest, largest = tbl[0], tbl[-1]
    corners = [(budgets[0], smallest), (budgets[0], largest),
               (budgets[-1], smallest), (budgets[-1], largest)]
    print(corners)
    print(f"{'C':>10} {'N':>11} {'examples':>10} {'feasible':>9}  val_loss")
    results = []
    for C, (dm, nl, N) in corners:
        ne = examples_for(C, N, sup_per_ex)
        print("ne",ne) 
        feasible = ne <= n_train
        if feasible:
            vl = train_isoflop_point(
                make_cfg(base_cfg, dm, nl), train_tok, train_mask,
                val_tok, val_mask, ne, batch_size, base_cfg.lr)
        else:
            vl = float("nan")
        results.append((C, N, ne, feasible, vl))
        print(f"{C:>10.1e} {N:>11,} {ne:>10,} {str(feasible):>9}  {vl:.4f}")

    losses = [r[4] for r in results if r[3]]
    if any(not r[3] for r in results):
        print("\n[!] Some corners INFEASIBLE (need repeats). Lower C_max.")
    elif len(losses) >= 2 and (max(losses) - min(losses)) < 0.05:
        print("\n[!] Corner losses nearly identical -> valleys likely flat "
              "(saturated or starved). Adjust budget range before full sweep.")
    else:
        print(f"\n[ok] Corners span {max(losses)-min(losses):.3f} loss -> "
              "grid has dynamic range. Proceed to full sweep.")
    return results

In [11]:
POOL_TOK, POOL_MASK = build_full_pool(cfg.max_seq_len, seed=0)
SUP_PER_EX = float(POOL_MASK.sum(axis=1).mean())
print(f"Average tokens per example: {SUP_PER_EX}")

Average tokens per example: 4.494599459945994


In [12]:
BATCH_SIZE = 256
VAL_SIZE = 20_000

SIZE_LADDER = [(64, 2), (96, 3), (128, 3), (192, 4), (256, 5), (384, 6)]
POOL_TOK, POOL_MASK = build_full_pool(cfg.max_seq_len, seed=0)
VAL_TOK,   VAL_MASK   = POOL_TOK[-VAL_SIZE:], POOL_MASK[-VAL_SIZE:]
TRAIN_TOK, TRAIN_MASK = POOL_TOK[:-VAL_SIZE], POOL_MASK[:-VAL_SIZE]

SUP_PER_EX = float(POOL_MASK.sum(axis=1).mean()) # average output tokens per example ≈ 4

# --- size ladder with real N ---
TBL = size_table(cfg, SIZE_LADDER)
N_MIN = min(N for *_, N in TBL)

# --- budget ceiling + working range ---
C_MAX = budget_ceiling(N_MIN, SUP_PER_EX, len(TRAIN_TOK))
C_MIN  = C_MAX / 2e1           # 3 decades; corner check will validate
BUDGETS = np.logspace(np.log10(C_MIN), np.log10(C_MAX), 5)
print(f"C_max={C_MAX:.2e}  C_min={C_MIN:.2e}")

# --- calibrate before committing ---
results = corner_check(cfg, TBL, BUDGETS, TRAIN_TOK, TRAIN_MASK,
                VAL_TOK, VAL_MASK, SUP_PER_EX, BATCH_SIZE)

C_max=2.65e+12  C_min=1.33e+11
[(np.float64(132677391723.78036), (64, 2, 100416)), (np.float64(132677391723.78036), (384, 6, 10634112)), (np.float64(2653547834475.6074), (64, 2, 100416)), (np.float64(2653547834475.6074), (384, 6, 10634112))]
         C           N   examples  feasible  val_loss
ne 48995
n_examples 48995
total steps 191
   1.3e+11     100,416     48,995      True  1.5989
ne 463
n_examples 463
total steps 1


ValueError: The cosine_decay_schedule requires positive decay_steps, got decay_steps=-49.

In [ ]:
# ----------------------------------------------------------------------
# 6. Full sweep: every (budget, size) point.
# ----------------------------------------------------------------------
import itertools
import pandas as pd

def run_sweep(base_cfg, tbl, budgets, train_tok, train_mask,
              val_tok, val_mask, sup_per_ex, batch_size):
    n_train = len(train_tok)
    rows = []
    for C, (dm, nl, N) in itertools.product(budgets, tbl):
        ne = examples_for(C, N, sup_per_ex)
        if ne > n_train or ne < batch_size:
            print(f"skip C={C:.1e} N={N:,} (ne={ne:,} out of range)")
            continue
        vl = train_isoflop_point(
            make_cfg(base_cfg, dm, nl), train_tok, train_mask,
            val_tok, val_mask, ne, batch_size, base_cfg.lr)
        D = ne * sup_per_ex
        rows.append({"C": C, "d_model": dm, "num_layers": nl, "N": N,
                     "n_examples": ne, "D": D, "val_loss": vl})
        print(f"C={C:.1e} N={N:>11,} ne={ne:>8,} -> val {vl:.4f}")
    return pd.DataFrame(rows)

# PART 2

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
from scipy.optimize import least_squares
from config import Config

BATCH_SIZE     = 256
SEED           = 0
VAL_SUP_TOKENS = 100_000      # fixed val set, counted in supervised tokens
MAX_LEN        = 14
MAX_N          = 1000         # operands 0..999

# model ladder: (d_model, num_layers) — the only two things varied
MODEL_LADDER = [(64, 2), (96, 3), (128, 3), (192, 4), (256, 5), (384, 6)]

# per-model-size peak LR, held fixed across D
PEAK_LR = {
    (64, 2):  1.5e-3,
    (96, 3):  1.2e-3,
    (128, 3): 1.0e-3,
    (192, 4): 8e-4,
    (256, 5): 6e-4,
    (384, 6): 5e-4,
}

# data ladder in SUPERVISED tokens, log-spaced out to the pool ceiling
# (~999,900 non-fundamental examples * ~4.6 sup-tok/ex, minus the val carve)
DATA_LADDER = [100_000, 250_000, 600_000, 1_400_000, 2_800_000, 4_300_000]

In [6]:
# ============================================================================
# 1. Pool construction, fixed val carve, nested prefixes
# ============================================================================
from data import create_example

def build_pool(seed=SEED):
    """All non-fundamental pairs, shuffled once. Fundamentals = both operands
    single-digit (a<10 and b<10) -> memorized lookup, excluded."""
    toks, masks = [], []
    for a in range(MAX_N):
        for b in range(MAX_N):
            if a < 10 and b < 10:
                continue
            t, m = create_example(a, b, MAX_LEN)
            toks.append(t)
            masks.append(m)
    toks  = np.asarray(toks,  dtype=np.int32)
    masks = np.asarray(masks, dtype=np.int32)
    perm = np.random.default_rng(seed).permutation(len(toks))
    return toks[perm], masks[perm]

def supervised_tokens(masks):
    return int(masks.sum())


def carve_val(pool_tok, pool_mask, val_sup_tokens=VAL_SUP_TOKENS):
    """Fixed val set taken from the front of the shuffled pool; the remainder
    is the training pool. Val is identical for every cell."""
    cum = np.cumsum(pool_mask.sum(axis=1))
    n_val = int(np.searchsorted(cum, val_sup_tokens) + 1)
    return (pool_tok[:n_val], pool_mask[:n_val],
            pool_tok[n_val:], pool_mask[n_val:])


def prefix_for_D(tr_tok, tr_mask, D_sup_tokens):
    """Nested prefix: fewest examples whose supervised tokens reach D."""
    cum = np.cumsum(tr_mask.sum(axis=1))
    n = min(int(np.searchsorted(cum, D_sup_tokens) + 1), len(tr_tok))
    return tr_tok[:n], tr_mask[:n]


def check_ladder(tr_mask, ladder=DATA_LADDER):
    """Verify every D is actually available; return the ceiling."""
    avail = supervised_tokens(tr_mask)
    print(f"train pool supervised tokens available: {avail:,}")
    for D in ladder:
        print(f"  D={D:>10,}  {'ok' if D <= avail else 'EXCEEDS POOL'}")
    if max(ladder) > avail:
        raise ValueError(
            f"largest D ({max(ladder):,}) exceeds pool ({avail:,}). "
            "Lower the top of DATA_LADDER.")
    return avail

In [ ]:
# ============================================================================
# 2. Config / param counting
# ============================================================================
from utils import active_params, check_param_formula

def make_cfg(base_cfg: Config, d_model, num_layers):
    """NOTE: if your Config does not derive heads from d_model, this keeps a
    fixed head count across the ladder. Adjust if you want key_dim pinned."""
    return base_cfg.replace(d_model=d_model, num_layers=num_layers)


def count_params(cfg: Config):
    """N for the 6ND compute model = ACTIVE params (3LDFK + 4DHL + 2DV).

    Counting every leaf would include the idle experts, which cost memory but
    no flops - at E=8/K=2 that inflates N by ~5.7x and makes the moe frontier
    incomparable to the dense one. See utils.active_params."""
    return active_params(cfg)


# the formula omits norm gammas and pos_embed; on a dense cfg the leftover should be
# small and constant-ish. run once so a Layer change can't silently desync N.
_probe = base_cfg.replace(num_experts=0)
_f, _a, _d = check_param_formula(_probe)
print(f"dense probe: formula {_f:,} vs pytree {_a:,} -> delta {_d:,} (norms + pos_embed)")
print(f"moe N (active) at base shape: {count_params(base_cfg):,}")

In [ ]:
# ============================================================================
# 3. Train one (N, D) cell — single epoch, matched cosine
# ============================================================================
import optax
from optax.losses import softmax_cross_entropy_with_integer_labels

def make_optimizer(peak_lr, total_steps, weight_decay=0.1):
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=peak_lr,
        # proportional warmup, capped at 50. The old max(50, ...) put 58% of a short
        # run into warmup, so small-D cells never trained at peak LR and flat-lined
        # at ~1.55 - which looked like a phase transition in the loss surface. It also
        # made decay_steps negative below 50 steps, killing those cells outright.
        warmup_steps=max(1, min(50, int(0.05 * total_steps))),
        decay_steps=total_steps,
        end_value=peak_lr * 0.1,
    )
    return optax.adamw(schedule, weight_decay=weight_decay)


def calc_val_loss(val_loader, weights, cfg):
    # report CE only - the lb term is a training regularizer, not part of the loss surface we fit
    losses = [loss_fn(weights, x, m, cfg)[1]['ce'].item() for x, m in val_loader]
    return float(np.mean(losses))

def loss_fn(weights: jax.Array, token_ids: jax.Array, mask: jax.Array, cfg: Config) -> jax.Array:
    logits, lb, _stats = forward(token_ids[:, :-1], weights, cfg)
    targets = token_ids[:, 1:]
    # for addition, we need to build a mask because the model needs to only predict the last 3 digits
    # need mask code
    loss_mask = mask[:, 1:]
    ce = softmax_cross_entropy_with_integer_labels(logits, targets) # mean would also change since masking the outputs
    ce = jnp.sum(ce * loss_mask) / jnp.sum(loss_mask)
    loss = ce + cfg.lb_factor * lb
    return loss, dict(ce = ce, lb = lb)


# ----------------------------------------------------------------------------
# 3b. Routing diagnostics — logged on EVERY run
# ----------------------------------------------------------------------------
# Without these a fit can look clean and be measuring the wrong thing: "moe scales
# worse at high E" and "my router collapsed at high E" produce the same loss curve.
# frac tells you batch-level balance, entropy tells you per-token confidence.
def collect_routing(weights, cfg, loader):
    """Average the per-layer router diagnostics over a loader. Returns None for a
    dense cfg (no router to report on)."""
    if cfg.num_experts == 0:
        return None

    @jax.jit
    def stats_for(x):
        _logits, _lb, stats = forward(x[:, :-1], weights, cfg)
        return stats

    frac_sum, ent_sum, n_batches = None, None, 0
    for x, _m in loader:
        stats = stats_for(x)
        f = np.stack([np.asarray(s['frac']) for s in stats])         # (L, E)
        e = np.array([float(s['entropy']) for s in stats])           # (L,)
        frac_sum = f if frac_sum is None else frac_sum + f
        ent_sum  = e if ent_sum  is None else ent_sum + e
        n_batches += 1

    frac = frac_sum / n_batches      # (L, E) fraction of dispatch slots per expert
    ent  = ent_sum / n_batches       # (L,)   nats
    uniform = 1.0 / cfg.num_experts

    return dict(
        frac=frac,                                   # full (L, E) table, kept for plots
        entropy=ent,
        max_frac=float(frac.max()),                  # 1/E balanced, ->1 collapsed
        min_frac=float(frac.min()),
        # share of experts receiving < 10% of their uniform allocation = effectively dead
        dead_frac=float((frac < 0.1 * uniform).mean()),
        mean_entropy=float(ent.mean()),
        # 1.0 = uniform routing, 0.0 = every token fully committed to one expert
        entropy_norm=float(ent.mean() / np.log(cfg.num_experts)),
        imbalance=float(frac.max() / uniform),       # 1.0 balanced, E = full collapse
    )


# Single-seed noise currently exceeds the trend: at D=600k the dense losses ran
# 0.62, 0.33, 1.54, 0.23, 0.13 as N increased - that 1.54 is a failed run, not a
# data point. Median over seeds is what makes the surface measurable.
# Cost scales linearly: set SEEDS = (0,) to go back to one run per cell.
SEEDS = (0, 1, 2)


def train_cell(base_cfg, d_model, num_layers, tr_tok, tr_mask,
               val_tok, val_mask, seeds=SEEDS):
    losses, routings = [], []
    for sd in seeds:
        vl, rt = _train_one(base_cfg, d_model, num_layers, tr_tok, tr_mask,
                            val_tok, val_mask, sd)
        losses.append(vl)
        if rt is not None:
            routings.append(rt)

    val_loss = float(np.median(losses))
    spread = float(max(losses) - min(losses))
    # routing diagnostics from the seed closest to the median loss
    routing = routings[int(np.argmin(np.abs(np.array(losses) - val_loss)))] if routings else None
    return val_loss, routing, spread, len(losses)


def _train_one(base_cfg, d_model, num_layers, tr_tok, tr_mask,
               val_tok, val_mask, seed=SEED):
    cfg = make_cfg(base_cfg, d_model, num_layers)
    total_steps = len(tr_tok) // BATCH_SIZE      # single epoch
    if total_steps < 1:
        raise ValueError(f"D too small for one batch: {len(tr_tok)} examples")

    weights = Weights.init(cfg, jax.random.key(seed))
    opt = make_optimizer(PEAK_LR[(d_model, num_layers)], total_steps)
    opt_state = opt.init(weights)

    @jax.jit
    def train_step(x, mask, weights, opt_state):
        # cfg is closed over, so it stays a python constant under jit (num_experts/top_k must be static)
        (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(weights, x, mask, cfg)
        updates, opt_state = opt.update(grads, opt_state, weights)
        weights = optax.apply_updates(weights, updates)
        return loss, aux, weights, opt_state

    for x, m in Dataloader(tr_tok, tr_mask, BATCH_SIZE):
        _, aux, weights, opt_state = train_step(x, m, weights, opt_state)

    val_loader = Dataloader(val_tok, val_mask, BATCH_SIZE, shuffle=False)
    val_loss = calc_val_loss(val_loader, weights, cfg)
    routing = collect_routing(
        weights, cfg, Dataloader(val_tok, val_mask, BATCH_SIZE, shuffle=False))
    return val_loss, routing

In [ ]:
# ============================================================================
# 4. Full-cross sweep over (E, N, D)
# ============================================================================
# E_LADDER includes 0 = dense, which is the reference arm every moe slice is
# compared against. Cost is len(E_LADDER) x len(MODEL_LADDER) x len(DATA_LADDER).
E_LADDER = [0, 2, 4, 8, 16]


def run_sweep(base_cfg, e_ladder=E_LADDER, seed=SEED):
    pool_tok, pool_mask = build_pool(seed)
    val_tok, val_mask, tr_tok, tr_mask = carve_val(pool_tok, pool_mask)

    print(f"pool: {len(pool_tok):,} ex, {supervised_tokens(pool_mask):,} sup-tok")
    print(f"val:  {len(val_tok):,} ex, {supervised_tokens(val_mask):,} sup-tok")
    D_ceiling = check_ladder(tr_mask)
    print(f"\ncells to run: {len(e_ladder)} E x {len(MODEL_LADDER)} N x {len(DATA_LADDER)} D "
          f"= {len(e_ladder)*len(MODEL_LADDER)*len(DATA_LADDER)}\n")

    rows = []
    for E in e_ladder:
        cfg_E = base_cfg.replace(num_experts=E)
        tag = "dense" if E == 0 else f"E={E}"
        print(f"--- {tag} " + "-" * 60)
        for (d_model, num_layers) in MODEL_LADDER:
            N = count_params(make_cfg(cfg_E, d_model, num_layers))   # ACTIVE params
            for D in DATA_LADDER:
                sub_tok, sub_mask = prefix_for_D(tr_tok, tr_mask, D)
                D_actual = supervised_tokens(sub_mask)
                val_loss, routing, spread, n_seeds = train_cell(
                    cfg_E, d_model, num_layers, sub_tok, sub_mask,
                    val_tok, val_mask)
                row = dict(n_experts=E, top_k=(cfg_E.top_k if E else 1),
                           d_model=d_model, num_layers=num_layers, N=N,
                           D_target=D, D_actual=D_actual, val_loss=val_loss,
                           seed_spread=spread, n_seeds=n_seeds)
                if routing is not None:
                    row.update(max_frac=routing['max_frac'],
                               min_frac=routing['min_frac'],
                               dead_frac=routing['dead_frac'],
                               mean_entropy=routing['mean_entropy'],
                               entropy_norm=routing['entropy_norm'],
                               imbalance=routing['imbalance'],
                               frac=routing['frac'])
                rows.append(row)

                diag = ""
                if routing is not None:
                    diag = (f"  | imbal={routing['imbalance']:.2f}"
                            f" H/logE={routing['entropy_norm']:.2f}"
                            f" dead={routing['dead_frac']:.0%}")
                print(f"  N={N:>9,}  D={D_actual:>9,}  val={val_loss:.4f}"
                      f" (+/-{spread:.3f} over {n_seeds}){diag}")
    return rows, D_ceiling


def rows_where(rows, n_experts):
    return [r for r in rows if r['n_experts'] == n_experts]


def routing_table(rows, e_ladder=E_LADDER):
    """Per-E routing health, worst case over the (N, D) grid. Read this BEFORE the
    fits: a slice with imbalance near E or dead experts is measuring collapse, and
    its A is not a scaling result."""
    print(f"{'E':>4} {'runs':>5} {'imbal max':>10} {'H/logE min':>11} "
          f"{'dead max':>9} {'verdict':>28}")
    for E in e_ladder:
        sub = [r for r in rows_where(rows, E) if 'imbalance' in r]
        if not sub:
            continue
        imb = max(r['imbalance'] for r in sub)
        hn  = min(r['entropy_norm'] for r in sub)
        dead = max(r['dead_frac'] for r in sub)
        if imb > 0.5 * E or dead > 0.25:
            verdict = "COLLAPSED - fit not valid"
        elif imb > 2.0 or dead > 0.0:
            verdict = "skewed - interpret with care"
        else:
            verdict = "healthy"
        print(f"{E:>4} {len(sub):>5} {imb:>10.2f} {hn:>11.3f} {dead:>9.1%} {verdict:>28}")

In [ ]:
import json, time

# --- persist ---------------------------------------------------------------
# The Colab VM is ephemeral: anything written to /content disappears when the
# runtime recycles (this is how the previous rows.pkl was lost). Copy to Drive.
def save_rows(rows, D_ceiling, tag):
    def _jsonable(r):
        return {k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()}

    stamp = time.strftime('%Y%m%d-%H%M%S')
    path = f"rows_{tag}_{stamp}.json"
    payload = dict(rows=[_jsonable(r) for r in rows], D_ceiling=D_ceiling,
                   tag=tag, stamp=stamp)
    with open(path, "w") as f:
        json.dump(payload, f)
    print(f"saved {len(rows)} rows -> {path}")

    if IN_COLAB:
        from google.colab import drive
        import shutil, os
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        dest_dir = '/content/drive/MyDrive/scaling-book'
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(path, dest_dir)
        print(f"copied -> {dest_dir}/{path}")
    return path

rows, D_ceiling = run_sweep(base_cfg)
ROWS_PATH = save_rows(rows, D_ceiling, 'moe')

# read this first - a collapsed router invalidates that E slice's fit
print()
routing_table(rows)


In [ ]:
# ---------------------------------------------------------------------------
# Reload a previous sweep instead of retraining. Point at a rows_moe_*.json
# (local or on Drive) and every fit/plot cell below works unchanged.
# ---------------------------------------------------------------------------
# import json
# with open("rows_moe_YYYYmmdd-HHMMSS.json") as f:
#     _payload = json.load(f)
# rows, D_ceiling = _payload["rows"], _payload["D_ceiling"]
# print(f"loaded {len(rows)} rows, D_ceiling={D_ceiling:,}")


In [26]:
# ============================================================================
# 5. Fit L(N,D) = E + A/N^alpha + B/D^beta  (Huber on log-loss, multi-restart)
# ============================================================================
import contextlib

@contextlib.contextmanager
def x64_fit():
    """Run the fitter in float64.

    _predict_logL is a jax function and least_squares differentiates it by finite
    differences. Under jax's default float32 the numerical jacobian is pure noise,
    least_squares terminates at its starting point, and fit_surface returns the
    SAME A and E_irr no matter what data you hand it - which silently destroys any
    per-E comparison. Scoped to the fit so training numerics are untouched.
    """
    old = jax.config.jax_enable_x64
    jax.config.update('jax_enable_x64', True)
    try:
        yield
    finally:
        jax.config.update('jax_enable_x64', old)


def _predict_logL(params, N, D):
    a, b, e, alpha, beta = params
    terms = jnp.stack([
        a - alpha * jnp.log(N),
        b - beta  * jnp.log(D),
        jnp.full_like(N, e),
    ], axis=0)
    return jax.scipy.special.logsumexp(terms, axis=0)


def fit_surface(rows, huber_delta=1e-3, n_restarts=30, seed=0):
    N = np.array([r['N'] for r in rows], dtype=np.float64)
    D = np.array([r['D_actual'] for r in rows], dtype=np.float64)
    logL = np.log(np.array([r['val_loss'] for r in rows], dtype=np.float64))
    Nj, Dj = jnp.asarray(N), jnp.asarray(D)

    def resid(p):
        return np.asarray(_predict_logL(p, Nj, Dj)) - logL

    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        p0 = np.array([rng.uniform(-2, 6),      # log A
                       rng.uniform(-2, 6),      # log B
                       rng.uniform(-2, 0),      # log E
                       rng.uniform(0.1, 0.9),   # alpha
                       rng.uniform(0.1, 0.9)])  # beta
        try:
            r = least_squares(resid, p0, loss='huber',
                              f_scale=huber_delta, max_nfev=10000)
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r

    if best is None:
        raise RuntimeError("all restarts failed")
    a, b, e, alpha, beta = best.x
    return dict(A=float(np.exp(a)), B=float(np.exp(b)), E=float(np.exp(e)),
                alpha=float(alpha), beta=float(beta),
                raw=best.x, cost=float(best.cost))


def bootstrap_fit(rows, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    keys = ['A', 'B', 'E', 'alpha', 'beta']
    samples = {k: [] for k in keys}
    extra = {'p': [], 'q': []}
    n = len(rows)
    for _ in range(n_boot):
        sub = [rows[i] for i in rng.integers(0, n, n)]
        try:
            f = fit_surface(sub, n_restarts=8, seed=int(rng.integers(1e9)))
        except Exception:
            continue
        for k in keys:
            samples[k].append(f[k])
        s = f['alpha'] + f['beta']
        extra['p'].append(f['beta'] / s)
        extra['q'].append(f['alpha'] / s)

    def ci(arr):
        arr = np.array(arr)
        return dict(median=float(np.median(arr)),
                    lo=float(np.percentile(arr, 2.5)),
                    hi=float(np.percentile(arr, 97.5)))
    out = {k: ci(samples[k]) for k in keys}
    out['p'] = ci(extra['p'])
    out['q'] = ci(extra['q'])
    return out

In [28]:
# ============================================================================
# 6. Part 2: impose compute analytically. D = C/(6N), minimize over N.
# ============================================================================
def derive_frontier(fit, C_grid=None, D_ceiling=None):
    """Closed form from stationarity of A/N^alpha + B/(C/6N)^beta:
       N_opt = [ (alpha*A)/(beta*B) * (C/6)^beta ]^(1/(alpha+beta))
       so N_opt ~ C^p, D_opt ~ C^q with p=beta/(alpha+beta), q=alpha/(alpha+beta)."""
    A, B, alpha, beta = fit['A'], fit['B'], fit['alpha'], fit['beta']
    p = beta / (alpha + beta)
    q = alpha / (alpha + beta)

    if C_grid is None:
        C_grid = np.logspace(11, 18, 50)

    N_opt = ((alpha * A) / (beta * B) * (C_grid / 6.0) ** beta) ** (1.0 / (alpha + beta))
    D_opt = C_grid / (6.0 * N_opt)
    L_opt = fit['E'] + A / N_opt ** alpha + B / D_opt ** beta

    grounded = C_star = None
    if D_ceiling is not None:
        grounded = D_opt <= D_ceiling
        C_star = float(C_grid[grounded].max()) if grounded.any() else None

    return dict(C=C_grid, N_opt=N_opt, D_opt=D_opt, L_opt=L_opt,
                p=p, q=q, grounded=grounded, C_max_grounded=C_star)

In [29]:

# ============================================================================
# 7. Report
# ============================================================================
def report(fit, ci, fr):
    print("\n=== fitted surface L(N,D) = E + A/N^alpha + B/D^beta ===")
    for k in ['A', 'B', 'E', 'alpha', 'beta']:
        c = ci[k]
        print(f"  {k:>5} = {fit[k]:.4g}   95% CI [{c['lo']:.4g}, {c['hi']:.4g}]")
    print("\n=== compute-optimal frontier ===")
    print(f"  N_opt ~ C^{fr['p']:.4f}   95% CI [{ci['p']['lo']:.4f}, {ci['p']['hi']:.4f}]")
    print(f"  D_opt ~ C^{fr['q']:.4f}   95% CI [{ci['q']['lo']:.4f}, {ci['q']['hi']:.4f}]")
    if fr['C_max_grounded'] is not None:
        print(f"\n  data ceiling reached at C = {fr['C_max_grounded']:.3g} FLOPs")
        print("  beyond this the frontier extrapolates past the dataset "
              "(data-constrained regime; cf. Muennighoff et al. 2023)")

In [ ]:
# dense reference arm (E=0). the moe slices are fitted per-E in section 9 below.
dense_rows = rows_where(rows, 0)
with x64_fit():
    fit = fit_surface(dense_rows)
    ci  = bootstrap_fit(dense_rows, n_boot=500)
fr  = derive_frontier(fit, D_ceiling=D_ceiling)
report(fit, ci, fr)

In [ ]:
# ============================================================================
# 8. Plot the frontier: empirical compute-optimal points vs the fitted power law
# ============================================================================
import matplotlib.pyplot as plt

INK, MUTED = "#0b0b0b", "#52514e"
MEASURED, FITLINE = "#2a78d6", "#eb6834"   # categorical slots 1 and 2


def frontier_points(rows, n_bins=6):
    """Empirical compute-optimal points. Each measured cell sits at its own
    C = 6ND, so bin the cells in log-C and take the lowest-loss cell per bin.
    A bin is only a real minimum if several model sizes land in it - n_cells
    is returned so thin bins can be judged (or dropped) rather than trusted."""
    N = np.array([r['N'] for r in rows], dtype=np.float64)
    D = np.array([r['D_actual'] for r in rows], dtype=np.float64)
    L = np.array([r['val_loss'] for r in rows], dtype=np.float64)
    C = 6.0 * N * D

    edges = np.logspace(np.log10(C.min()), np.log10(C.max()) + 1e-9, n_bins + 1)
    which = np.clip(np.digitize(C, edges) - 1, 0, n_bins - 1)

    pts = []
    for b in range(n_bins):
        sel = np.flatnonzero(which == b)
        if sel.size == 0:
            continue
        j = sel[np.argmin(L[sel])]
        pts.append(dict(C=C[j], N=N[j], D=D[j], L=L[j], n_cells=int(sel.size)))
    return pts, dict(C=C, N=N, D=D, L=L)


def plot_frontier(rows, fit, fr, n_bins=6):
    pts, cloud = frontier_points(rows, n_bins)
    c_lo, c_hi = cloud['C'].min(), cloud['C'].max()

    # the fit is only grounded where we actually measured - split the curve there
    inside = (fr['C'] >= c_lo) & (fr['C'] <= c_hi)
    panels = [
        ("N_opt", 'N', 'N_opt', f"N ~ C^{fr['p']:.3f}", "parameters"),
        ("D_opt", 'D', 'D_opt', f"D ~ C^{fr['q']:.3f}", "supervised tokens"),
        ("L_opt", 'L', 'L_opt', "fitted L(N,D) along the frontier", "val loss (CE)"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, (_, cloud_key, fit_key, fitlabel, ylab) in zip(axes, panels):
        # every measured cell, recessive - context for how the minima were found
        ax.scatter(cloud['C'], cloud[cloud_key], s=14, color=MEASURED, alpha=0.22,
                   linewidths=0, zorder=2, label="all measured cells")

        # fitted frontier: solid where grounded, dashed where extrapolated
        ax.plot(fr['C'][inside], fr[fit_key][inside], color=FITLINE, lw=2,
                zorder=3, label=fitlabel)
        ax.plot(fr['C'][~inside], fr[fit_key][~inside], color=FITLINE, lw=2,
                ls=(0, (4, 3)), alpha=0.55, zorder=3, label="extrapolated")

        # empirical minima - white ring so they read on top of the cloud
        ax.scatter([p['C'] for p in pts], [p[cloud_key] for p in pts], s=64,
                   color=MEASURED, edgecolor="white", linewidth=2, zorder=4,
                   label="per-budget best cell")

        ax.set_xscale('log')
        if cloud_key != 'L':
            ax.set_yscale('log')
        # keep the view on the measured range - fr's default C_grid runs to 1e18
        # and would otherwise squeeze every measured point into the left edge
        ax.set_xlim(c_lo / 3, c_hi * 30)
        ax.set_xlabel("compute C (FLOPs)", color=MUTED, fontsize=10)
        ax.set_ylabel(ylab, color=MUTED, fontsize=10)
        ax.grid(True, which='major', color=MUTED, alpha=0.12, lw=0.8)
        ax.set_axisbelow(True)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color(MUTED)
            ax.spines[side].set_alpha(0.4)
        ax.tick_params(colors=MUTED, labelsize=9)

        # data ceiling: past here the frontier wants more data than the pool has
        if fr.get('C_max_grounded'):
            ax.axvline(fr['C_max_grounded'], color=MUTED, lw=1,
                       ls=(0, (2, 3)), alpha=0.6, zorder=1)

    axes[0].set_title(f"compute-optimal frontier  (E={fit['E']:.3f}, "
                      f"alpha={fit['alpha']:.3f}, beta={fit['beta']:.3f})",
                      color=INK, fontsize=12, loc='left', pad=12)
    axes[0].legend(frameon=False, fontsize=9, labelcolor=MUTED, loc='best')

    fig.tight_layout()
    plt.show()

    print(f"{'C':>10} {'N':>11} {'D':>11} {'val_loss':>9} {'cells in bin':>13}")
    for p in pts:
        print(f"{p['C']:>10.2e} {p['N']:>11,.0f} {p['D']:>11,.0f} "
              f"{p['L']:>9.4f} {p['n_cells']:>13}")
    if fr.get('C_max_grounded'):
        print(f"\ndotted vline = data ceiling at C={fr['C_max_grounded']:.3g} "
              "(frontier beyond it needs more data than the pool holds)")


# dense reference arm; the per-E moe frontiers are plotted in section 11
plot_frontier(dense_rows, fit, fr)

In [31]:
def diagnose_bootstrap(rows, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    a_s, b_s, e_s = [], [], []
    n = len(rows)
    for _ in range(n_boot):
        sub = [rows[i] for i in rng.integers(0, n, n)]
        try:
            f = fit_surface(sub, n_restarts=8, seed=int(rng.integers(1e9)))
        except Exception:
            continue
        a_s.append(f['alpha']); b_s.append(f['beta']); e_s.append(f['E'])
    a_s, b_s, e_s = map(np.array, (a_s, b_s, e_s))
    for name, arr in [('alpha', a_s), ('beta', b_s)]:
        print(f"{name}: median {np.median(arr):.3f}  "
              f"IQR [{np.percentile(arr,25):.3f}, {np.percentile(arr,75):.3f}]  "
              f"frac > 1.0: {(arr > 1.0).mean():.3f}")
    print(f"corr(alpha, log E) = {np.corrcoef(a_s, np.log(e_s))[0,1]:.3f}")
    return a_s, b_s, e_s

with x64_fit():
    diagnose_bootstrap(dense_rows)

alpha: median 0.678  IQR [0.502, 0.870]  frac > 1.0: 0.208
beta: median 0.509  IQR [0.368, 0.686]  frac > 1.0: 0.066
corr(alpha, log E) = 0.069


(array([ 0.39165282,  0.49073114,  0.84525637,  0.2798987 ,  0.66969239,
         0.74236049,  4.80755757,  0.50784443,  0.82499613,  5.70514305,
         0.73954958,  2.72119493,  0.69684261,  0.32847263,  0.4014916 ,
         0.37812599,  0.77776391,  0.89750646,  0.74628067,  0.71326075,
         0.57259626,  0.87714356,  0.7992693 ,  0.76564478,  0.55822344,
         0.22494558,  0.53950164,  0.16333883,  0.35092803,  0.35372145,
         0.61249263,  0.38945382,  0.76889691,  0.55542824,  0.52227936,
         0.41640593,  0.75849392,  0.69827374,  0.88315844,  0.5522495 ,
         0.47742872,  7.15426099,  0.52698735,  0.54508528,  0.75327946,
         3.14668503,  5.38486106,  0.28704215,  0.81642963,  0.6848769 ,
         4.51306199,  0.37797413,  0.72264052,  0.59203074,  5.42771508,
         0.69833226,  0.40311345,  0.80396099,  0.89861665,  0.87191872,
         0.62545308,  0.6326403 ,  0.4226541 ,  0.8897161 ,  0.68140421,
         5.45028753,  0.27846932,  0.6384528 ,  0.7

In [ ]:
# ============================================================================
# 9. Per-E fits — same fitter, called once per slice
# ============================================================================
# fit_surface is unchanged: same log-sum-exp parameterization, same huber delta,
# same multi-restart grid. Only the row subset differs.
#
# What to look for in the table:
#   alpha, beta, B   roughly flat across E   -> the surface shape is E-invariant
#   E_irr (c)        near zero everywhere    -> no spurious floor
#   A                declining with E        -> the moe effect lives entirely in A
# E_irr is the irreducible term of L = E_irr + A/N^alpha + B/D^beta. It is named
# 'E' inside fit_surface; renamed here so it cannot be confused with num_experts.

with x64_fit():
    fits = {E: fit_surface(rows_where(rows, E)) for E in E_LADDER}
    cis  = {E: bootstrap_fit(rows_where(rows, E), n_boot=200) for E in E_LADDER}


def fit_table(fits, cis, e_ladder=E_LADDER):
    print(f"{'E':>4} {'n':>4} {'A':>12} {'B':>12} {'E_irr (c)':>12} "
          f"{'alpha':>8} {'beta':>8}")
    for E in e_ladder:
        f, c = fits[E], cis[E]
        n = len(rows_where(rows, E))
        print(f"{E:>4} {n:>4} {f['A']:>12.4g} {f['B']:>12.4g} {f['E']:>12.4g} "
              f"{f['alpha']:>8.4f} {f['beta']:>8.4f}")
    print()
    print("95% CI (bootstrap):")
    for E in e_ladder:
        c = cis[E]
        print(f"  E={E:<3} A [{c['A']['lo']:.4g}, {c['A']['hi']:.4g}]   "
              f"alpha [{c['alpha']['lo']:.3f}, {c['alpha']['hi']:.3f}]   "
              f"beta [{c['beta']['lo']:.3f}, {c['beta']['hi']:.3f}]")

    # the three flatness checks, stated numerically rather than eyeballed
    print()
    moe_E = [E for E in e_ladder if E > 0]
    for key in ('alpha', 'beta', 'B'):
        vals = np.array([fits[E][key] for E in moe_E])
        spread = vals.max() / max(vals.min(), 1e-12)
        flag = "flat" if spread < 1.5 else "NOT flat - pooling is unjustified"
        print(f"  {key:>6}: {vals.round(4)}  max/min={spread:.2f}  {flag}")
    A_vals = np.array([fits[E]['A'] for E in moe_E])
    mono = np.all(np.diff(A_vals) < 0)
    print(f"  {'A':>6}: {A_vals.round(4)}  "
          f"{'monotonically declining in E' if mono else 'NOT monotonic - second stage is not meaningful'}")
    c_vals = np.array([fits[E]['E'] for E in e_ladder])
    print(f"  {'E_irr':>6}: {c_vals.round(4)}  "
          f"{'near zero' if np.all(c_vals < 0.05) else 'nonzero - a floor is being absorbed here'}")


fit_table(fits, cis)

In [ ]:
# ============================================================================
# 10. Second stage: A(E) = a + g / E^gamma
# ============================================================================
# Run this ONLY if section 9 showed alpha/beta/B flat and A declining. Four points
# and three parameters is thin - the fit is reported with error bars, not as a law.

def fit_A_of_E(fits, cis, e_ladder=None, n_restarts=200, seed=0):
    """1-D least squares of A(E) = a + g/E^gamma over the moe slices.

    Weighted by each A's bootstrap CI width, so a slice the first-stage fitter was
    unsure about does not drag the curve. Returns the fit plus the diagnostics
    needed to decide whether to believe it."""
    e_ladder = e_ladder or [E for E in fits if E > 0]
    Evals = np.array(sorted(e_ladder), dtype=np.float64)
    A = np.array([fits[int(E)]['A'] for E in Evals], dtype=np.float64)
    # CI half-width as sigma; floor it so a degenerate CI cannot dominate
    sigma = np.array([max((cis[int(E)]['A']['hi'] - cis[int(E)]['A']['lo']) / 2.0,
                          1e-3 * abs(fits[int(E)]['A'])) for E in Evals])

    n_pts, n_par = len(Evals), 3
    def resid(p):
        a, g, gamma = p
        return (a + g / Evals ** gamma - A) / sigma

    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        p0 = np.array([rng.uniform(0, max(A.max(), 1e-6)),
                       rng.uniform(0, 5 * max(A.max(), 1e-6)),
                       rng.uniform(0.05, 2.0)])
        try:
            r = least_squares(resid, p0, max_nfev=10000)
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r
    if best is None:
        raise RuntimeError("all restarts failed")

    a, g, gamma = best.x
    pred = a + g / Evals ** gamma
    ss_res = float(np.sum((A - pred) ** 2))
    ss_tot = float(np.sum((A - A.mean()) ** 2))
    dof = n_pts - n_par

    # parameter sigmas from the jacobian; only meaningful when dof > 0
    perr = np.full(3, np.nan)
    if dof > 0:
        try:
            _, s, VT = np.linalg.svd(best.jac, full_matrices=False)
            s = s[s > 1e-12]
            cov = (VT[:len(s)].T / s**2) @ VT[:len(s)]
            perr = np.sqrt(np.diag(cov) * (2 * best.cost / dof))
        except Exception:
            pass

    return dict(a=float(a), g=float(g), gamma=float(gamma),
                perr=perr, E=Evals, A=A, sigma=sigma, pred=pred,
                r2=float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan'),
                dof=dof, n_points=n_pts)


def report_A_of_E(af):
    lo_dof = af['dof'] <= 0
    print("=== second stage: A(E) = a + g / E^gamma ===")
    print(f"  points: {af['n_points']}   parameters: 3   dof: {af['dof']}")
    if lo_dof:
        print("  [!] dof <= 0 - the curve interpolates the points and the error bars are\n"
              "      undefined. Add E=32 before quoting gamma as anything.")
    names = ('a', 'g', 'gamma')
    for i, nm in enumerate(names):
        e = af['perr'][i]
        pm = f" +/- {e:.4g}" if np.isfinite(e) else "  (undefined)"
        print(f"  {nm:>6} = {af[nm]:.6g}{pm}")
    print(f"  R^2 = {af['r2']:.4f}")
    print()
    print(f"  {'E':>4} {'A fitted':>12} {'A(E) pred':>12} {'resid':>10} {'sigma':>10}")
    for E, a_i, p_i, s_i in zip(af['E'], af['A'], af['pred'], af['sigma']):
        print(f"  {int(E):>4} {a_i:>12.5g} {p_i:>12.5g} {a_i - p_i:>10.3g} {s_i:>10.3g}")
    print()
    print(f"  asymptote a = {af['a']:.5g}: the A that infinite experts would buy.")
    print("  Treat gamma as indicative. With 4 points it is not separately identified\n"
          "  from g - report the curve, not the exponent.")


def plot_A_of_E(af):
    fig, ax = plt.subplots(figsize=(6, 4.2))
    grid = np.logspace(np.log10(af['E'].min()), np.log10(af['E'].max()), 200)
    ax.plot(grid, af['a'] + af['g'] / grid ** af['gamma'], color=FITLINE, lw=2,
            zorder=2, label=f"a + g/E^{af['gamma']:.2f}")
    ax.axhline(af['a'], color=MUTED, lw=1, ls=(0, (2, 3)), alpha=0.7, zorder=1)
    ax.errorbar(af['E'], af['A'], yerr=af['sigma'], fmt='o', color=MEASURED,
                ecolor=MEASURED, elinewidth=1.5, capsize=4, markersize=7,
                markeredgecolor='white', markeredgewidth=1.5, zorder=3,
                label="per-E fitted A (95% CI)")
    ax.set_xscale('log'); ax.set_xticks(af['E'])
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    # E is a discrete ladder - suppress the log decade minor ticks between the points
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    ax.set_xlabel("num_experts E", color=MUTED, fontsize=10)
    ax.set_ylabel("A", color=MUTED, fontsize=10)
    ax.set_title(f"A declines with E   (asymptote a={af['a']:.4g}, R^2={af['r2']:.3f})",
                 color=INK, fontsize=11, loc='left', pad=10)
    ax.grid(True, color=MUTED, alpha=0.12, lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(MUTED); ax.spines[s].set_alpha(0.4)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.legend(frameon=False, fontsize=9, labelcolor=MUTED)
    fig.tight_layout(); plt.show()


import matplotlib
A_fit = fit_A_of_E(fits, cis)
report_A_of_E(A_fit)
plot_A_of_E(A_fit)

In [ ]:
# ============================================================================
# 11. The optimum, with A(E) in place of the scalar A
# ============================================================================
#   N_opt(C, E) = [ alpha * A(E) * C^beta / (beta * B * kappa^beta) ]^(1/(alpha+beta))
#   D_opt(C, E) = C / (kappa * N_opt)
#
# kappa = 6 as in the dense case: still a good approximation for moe flops once N
# is ACTIVE params, since only top_k experts run per token.
#
# The exponent p = beta/(alpha+beta) does not depend on A, so every E line has the
# SAME slope in log-log as dense. Only the intercept moves. That is the whole moe
# effect in one sentence, and it is the sentence worth putting in the writeup.

KAPPA = 6.0


def pooled_shape(fits, e_ladder=None):
    """alpha, beta, B shared across E (section 9 is what licenses this). Median is
    used rather than the mean so one bad slice cannot drag the pool."""
    e_ladder = e_ladder or [E for E in fits if E > 0]
    out = {k: float(np.median([fits[E][k] for E in e_ladder]))
           for k in ('alpha', 'beta', 'B', 'E')}
    out['spread'] = {k: (float(np.min([fits[E][k] for E in e_ladder])),
                         float(np.max([fits[E][k] for E in e_ladder])))
                     for k in ('alpha', 'beta', 'B')}
    return out


def A_of_E(af, E):
    return af['a'] + af['g'] / np.asarray(E, dtype=np.float64) ** af['gamma']


def frontier_for_A(A_val, shape, C_grid, kappa=KAPPA, D_ceiling=None):
    """Closed form with A supplied explicitly, so it serves both the dense arm
    (A from its own fit) and each moe E (A from A(E))."""
    alpha, beta, B = shape['alpha'], shape['beta'], shape['B']
    p = beta / (alpha + beta)
    q = alpha / (alpha + beta)

    N_opt = (alpha * A_val * C_grid ** beta / (beta * B * kappa ** beta)) ** (1.0 / (alpha + beta))
    D_opt = C_grid / (kappa * N_opt)
    L_opt = shape['E'] + A_val / N_opt ** alpha + B / D_opt ** beta

    C_star = None
    if D_ceiling is not None:
        # first C at which the frontier wants more tokens than the pool holds
        over = D_opt > D_ceiling
        if over.all():
            C_star = float(C_grid[0])   # already over the ceiling at the left edge
        elif over.any():
            C_star = float(C_grid[over][0])

    return dict(C=C_grid, N_opt=N_opt, D_opt=D_opt, L_opt=L_opt,
                ratio=D_opt / N_opt, p=p, q=q, C_cross=C_star, A=float(A_val))


def build_frontiers(fits, A_fit, D_ceiling, C_grid=None, e_ladder=None):
    e_ladder = e_ladder or [E for E in fits if E > 0]
    shape = pooled_shape(fits, e_ladder)
    if C_grid is None:
        C_grid = np.logspace(11, 18, 400)

    out = {}
    # dense reference uses its own fitted A and its own shape - it is the control,
    # not a point on the A(E) curve
    dense_shape = {k: fits[0][k] for k in ('alpha', 'beta', 'B', 'E')}
    out[0] = frontier_for_A(fits[0]['A'], dense_shape, C_grid, D_ceiling=D_ceiling)
    for E in e_ladder:
        out[E] = frontier_for_A(float(A_of_E(A_fit, E)), shape, C_grid,
                                D_ceiling=D_ceiling)
    return out, shape


def report_frontiers(frs, shape, A_fit, D_ceiling, e_ladder=None):
    e_ladder = e_ladder or [E for E in frs if E > 0]
    print("=== pooled surface shape (moe slices) ===")
    for k in ('alpha', 'beta', 'B'):
        lo, hi = shape['spread'][k]
        print(f"  {k:>6} = {shape[k]:.4f}   across-E range [{lo:.4g}, {hi:.4g}]")
    print(f"\n  p = beta/(alpha+beta) = {frs[e_ladder[0]]['p']:.4f}  "
          f"(N_opt ~ C^p, identical for every E by construction)")
    print(f"  q = alpha/(alpha+beta) = {frs[e_ladder[0]]['q']:.4f}  (D_opt ~ C^q)")
    print(f"\n  dense p = {frs[0]['p']:.4f} from its own fit - if this differs much from\n"
          f"  the moe p, the 'same slope, different intercept' claim does not hold.")

    print(f"\n=== tokens per active param, and where each line leaves the data ===")
    print(f"  D ceiling = {D_ceiling:,} supervised tokens")
    print(f"  {'E':>5} {'A':>12} {'D/N at C=1e13':>15} {'C at ceiling':>14}")
    for E in [0] + list(e_ladder):
        fr = frs[E]
        j = int(np.argmin(np.abs(fr['C'] - 1e13)))
        cross = f"{fr['C_cross']:.3g}" if fr['C_cross'] else "not reached"
        label = "dense" if E == 0 else f"E={E}"
        print(f"  {label:>5} {fr['A']:>12.4g} {fr['ratio'][j]:>15.2f} {cross:>14}")

In [ ]:
# ============================================================================
# 12. The plot: tokens per active param (D_opt / N_opt) vs compute, one line per E
# ============================================================================
# If the theory holds, higher E sits ABOVE dense: more tokens per active param.
# Each line is solid only up to the point where its D_opt crosses the data ceiling;
# past that marker it is extrapolation that cannot be validated here, and it is
# drawn dashed and said out loud rather than quietly continued.

# categorical slots 1-4 for the E lines (validated as a set), neutral for dense.
E_COLORS = {2: "#2a78d6", 4: "#eb6834", 8: "#1baf7a", 16: "#eda100"}
DENSE_COLOR = "#52514e"


def plot_tokens_per_param(frs, D_ceiling, e_ladder=None, C_lo=1e11, C_hi=1e17):
    e_ladder = e_ladder or sorted(E for E in frs if E > 0)
    fig, ax = plt.subplots(figsize=(9.5, 5.6))

    def draw(E, fr, color, label, lw, z):
        C, ratio = fr['C'], fr['ratio']
        vis = (C >= C_lo) & (C <= C_hi)
        cross = fr['C_cross']
        grounded = vis & (C <= cross) if cross else vis

        ax.plot(C[grounded], ratio[grounded], color=color, lw=lw, zorder=z, label=label)
        if cross is not None:
            beyond = vis & (C >= cross)
            ax.plot(C[beyond], ratio[beyond], color=color, lw=lw, ls=(0, (4, 3)),
                    alpha=0.5, zorder=z)
            j = int(np.argmin(np.abs(C - cross)))
            # marker = the last compute budget this dataset can actually validate
            ax.plot([cross], [ratio[j]], marker='o', markersize=8, color=color,
                    markeredgecolor='white', markeredgewidth=2, zorder=z + 1)

        # remember the right-edge value; labels are placed after de-collision below
        k = np.flatnonzero(vis)[-1]
        label_anchors.append((ratio[k], label))

    label_anchors = []
    draw(0, frs[0], DENSE_COLOR, "dense", 2.0, 3)
    for E in e_ladder:
        draw(E, frs[E], E_COLORS.get(E, MEASURED), f"E={E}", 2.0, 4)

    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(C_lo, C_hi * 3)
    ax.set_xlabel("compute C (FLOPs)", color=MUTED, fontsize=10)
    ax.set_ylabel("D_opt / N_opt  (supervised tokens per active param)",
                  color=MUTED, fontsize=10)
    ax.set_title("Compute-optimal tokens per active parameter",
                 color=INK, fontsize=12, loc='left', pad=26)
    ax.text(0.0, 1.02, "solid = grounded in measured data   |   dashed = past the "
            f"{D_ceiling/1e6:.1f}M-token ceiling, unvalidated",
            transform=ax.transAxes, fontsize=9, color=MUTED)

    # direct labels at the right edge - required relief for the low-contrast slots,
    # and it keeps identity off color alone. Lines converge at high C, so nudge the
    # labels apart in axes space rather than letting them overprint.
    lo, hi = np.log10(ax.get_ylim())
    placed = sorted(((np.log10(v) - lo) / (hi - lo), t) for v, t in label_anchors)
    MIN_GAP = 0.045
    for i in range(1, len(placed)):
        if placed[i][0] - placed[i - 1][0] < MIN_GAP:
            placed[i] = (placed[i - 1][0] + MIN_GAP, placed[i][1])
    for frac, text in placed:
        ax.annotate(text, xy=(1.005, frac), xycoords='axes fraction',
                    va='center', fontsize=9, color=MUTED, annotation_clip=False)
    ax.grid(True, which='major', color=MUTED, alpha=0.12, lw=0.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(MUTED); ax.spines[s].set_alpha(0.4)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.legend(frameon=False, fontsize=9, labelcolor=MUTED, loc='lower left')
    fig.tight_layout(); plt.show()

    print(f"{'':>6} {'C at ceiling':>14} {'D/N there':>11}  everything right of this is extrapolation")
    for E in [0] + list(e_ladder):
        fr = frs[E]
        label = "dense" if E == 0 else f"E={E}"
        if fr['C_cross'] is None:
            print(f"{label:>6} {'not reached':>14} {'-':>11}")
        else:
            j = int(np.argmin(np.abs(fr['C'] - fr['C_cross'])))
            print(f"{label:>6} {fr['C_cross']:>14.3g} {fr['ratio'][j]:>11.2f}")


frs, shape = build_frontiers(fits, A_fit, D_ceiling)
report_frontiers(frs, shape, A_fit, D_ceiling)
plot_tokens_per_param(frs, D_ceiling)